# NEXUS B2B — Self-Hosted OpenAI-Compatible LLM Server (Colab Pro A100 80GB)

- **Model**: `Qwen/Qwen2.5-14B-Instruct` (strong Arabic `ar-EG` + English + strict JSON Schema & tool calling, ~28 GB VRAM in BF16 on A100 80GB)
- **Inference Engine**: `vLLM` (OpenAI-compatible `/v1/chat/completions`)
- **Reverse Proxy & Auth**: `Nginx` on port `8080` enforcing `Authorization: Bearer <API_KEY>`
- **Public HTTPS URL**: `cloudflared` quick tunnel (no account required)

In [ ]:
# Cell 1: Verify A100 GPU & install vLLM + Nginx + cloudflared
!nvidia-smi
!pip install -q vllm
!apt-get update -qq && apt-get install -y -qq nginx
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print('Installation complete.')

In [ ]:
# Cell 2: Generate API Key & configure Nginx reverse proxy with Bearer auth
import secrets
from pathlib import Path

API_KEY = 'nexus-sk-' + secrets.token_urlsafe(24)
MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct-AWQ'

nginx_conf = f'''
server {{
    listen 8081;
    server_name _;

    location / {{
        if ($http_authorization != "Bearer {API_KEY}") {{
            return 401 '{{"error": {{"message": "Unauthorized", "type": "invalid_api_key"}}}}';
        }}
        proxy_pass http://127.0.0.1:8000;
        proxy_http_version 1.1;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_read_timeout 300s;
        proxy_send_timeout 300s;
    }}
}}
'''

Path('/etc/nginx/sites-available/nexus-llm').write_text(nginx_conf)
!ln -sf /etc/nginx/sites-available/nexus-llm /etc/nginx/sites-enabled/nexus-llm
!rm -f /etc/nginx/sites-enabled/default
!pkill -9 nginx 2>/dev/null || true
!nginx -t && nginx
print('Nginx running on :8081 | API Key:', API_KEY)

In [ ]:
# Cell 3: Start vLLM + cloudflared tunnel and keep running forever
import os
import re
import subprocess
import time
import urllib.request

vllm_cmd = [
    'vllm', 'serve', MODEL_ID,
    '--host', '127.0.0.1',
    '--port', '8000',
    '--dtype', 'half',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.90',
    '--enforce-eager',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
]
vllm_env = {**os.environ, 'VLLM_USE_V1': '0'}
vllm_log = open('/tmp/vllm.log', 'w')
vllm_proc = subprocess.Popen(vllm_cmd, env=vllm_env, stdout=vllm_log, stderr=subprocess.STDOUT)

print(f'Loading {MODEL_ID} in vLLM (downloading ~8.5 GB on first run)...')
while True:
    if vllm_proc.poll() is not None:
        print(Path('/tmp/vllm.log').read_text()[-6000:])
        raise RuntimeError('vLLM exited unexpectedly')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                print('vLLM is ready!')
                break
    except Exception:
        lines = Path('/tmp/vllm.log').read_text().strip().splitlines()
        if lines:
            print('[vLLM]', lines[-1][:140])
        time.sleep(10)

tunnel_log = open('/tmp/cloudflared.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--protocol', 'http2', '--url', 'http://127.0.0.1:8081', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    text = Path('/tmp/cloudflared.log').read_text()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print(Path('/tmp/cloudflared.log').read_text()[-2000:])
    raise RuntimeError('Could not obtain trycloudflare URL.')

print('Verifying public tunnel...', public_url)
verified = False
for _ in range(15):
    try:
        req = urllib.request.Request(
            f'{public_url}/v1/models',
            headers={'Authorization': f'Bearer {API_KEY}'},
        )
        with urllib.request.urlopen(req, timeout=5) as r:
            if r.status == 200:
                verified = True
                break
    except Exception:
        time.sleep(2)

if not verified:
    print(Path('/tmp/cloudflared.log').read_text()[-2000:])
    raise RuntimeError('Tunnel URL did not return 200 OK')

print('\n' + '=' * 60)
print('COPY THESE LINES INTO /var/www/html/Nexus-B2B/.env :')
print('=' * 60)
print('USE_MOCK_LLM=false')
print('LLM_PROVIDER=openai_compatible')
print(f'LLM_BASE_URL={public_url}/v1')
print(f'LLM_API_KEY={API_KEY}')
print(f'LLM_MODEL_SMART={MODEL_ID}')
print(f'LLM_MODEL_FAST={MODEL_ID}')
print('LLM_REASONING_MODELS=')
print('=' * 60)

# Keep notebook alive forever while serving requests
try:
    while True:
        if vllm_proc.poll() is not None:
            print('vLLM stopped! Last log:', Path('/tmp/vllm.log').read_text()[-1000:])
            break
        if tunnel_proc.poll() is not None:
            print('cloudflared stopped! Last log:', Path('/tmp/cloudflared.log').read_text()[-1000:])
            break
        time.sleep(30)
except KeyboardInterrupt:
    tunnel_proc.terminate()
    vllm_proc.terminate()
    print('Stopped.')